# Defect Formation Energy of Charged Defects

Calculate the formation energy of a defect as a function of its charge state and of the supercell size, with a multi-material DFT workflow on the Mat3ra platform. For the neutral defect in a single supercell see [Defect Formation Energy](defect_formation_energy.ipynb).

One job is created per (supercell size, charge state) pair. Each job takes **two materials, in order**:

- **[0] Defective supercell** — the pristine supercell with `DEFECT_CONFIGS` applied.
- **[1] Pristine supercell** — the defect-free supercell of the same size.

The workflow reports the raw formation energy of the charged cell,

$$E_f^{\text{raw}}[X^q] = E_{\text{tot}}[X^q] - E_{\text{tot}}[\text{bulk}] - \sum_i \Delta N_i\, \mu_i \quad [\text{eV}]$$

and this notebook adds the electron-reservoir term, giving the formation energy as a function of the electron chemical potential $\mu_e$ measured from the valence band maximum:

$$E_f[X^q](\mu_e) = E_f^{\text{raw}}[X^q] + q\,(E_{\text{VBM}} + \mu_e), \qquad 0 \le \mu_e \le E_{\text{gap}}$$

$E_{\text{VBM}}$ and $E_{\text{gap}}$ come from a Band Gap job on the pristine supercell; $\mu_i$ from the Standata elemental reference materials, as in [Formation Energy](formation_energy.ipynb). The charge $q$ enters as `tot_charge` in the QE `&SYSTEM` namelist and is compensated by a uniform jellium background.

No analytical finite-size correction (Freysoldt-Neugebauer-Van de Walle, Makov-Payne) is applied. Running several `SUPERCELL_SCALINGS` and extrapolating $E_f(L\to\infty)$ from the fitted image-charge terms takes its place; a single size gives the uncorrected value for that cell.

<h2 style="color:green">Usage</h2>

1. Set the pristine material, the defects, the supercell sizes and the charge states in cells 1.2 and 1.3 below.
1. Click "Run" > "Run All".
1. Total Energy and Band Gap jobs are created for any pristine supercell that lacks them, then one Defect Formation Energy job per (size, charge).
1. Scroll down for the results table, the formation energy against the electron chemical potential with the stable charge states, and the finite-size extrapolation.

## Summary

1. Set up the environment and parameters.
1. Authenticate and initialize API client.
1. Build a pristine/defective supercell pair per size, resolve elemental references, and save them.
1. Configure the Defect Formation Energy workflow per charge state and the pristine reference workflows.
1. Configure compute.
1. Create, submit, and monitor the pristine reference jobs and one defect job per (size, charge).
1. Retrieve results: raw formation energies, the dependence on the electron chemical potential, and the finite-size extrapolation.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
PRISTINE_NAME = "Si"  # defect-free cell, loaded from FOLDER or from Standata by name
VISUALIZATION_REPETITIONS = [1, 1, 1]

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Defect Formation Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific charge state and supercell size parameters

In [ ]:
# Defects placed in every supercell, in crystal coordinates of the pristine cell.
# See create_point_defect.ipynb for the full defect configuration reference.
DEFECT_CONFIGS = [
    {"type": "vacancy", "coordinate": [0.0, 0.0, 0.0], "placement_method": "closest_site"},
]

# Supercell sizes: n -> n x n x n repetitions of the pristine cell. Two or more
# sizes enable the finite-size extrapolation in section 7.3.
SUPERCELL_SCALINGS = [1]  # e.g. [2, 3]

# Net charge q of the defective supercell in units of e, one job per value.
CHARGES = [0]  # e.g. [1, 0, -1, -2, -3]

# K-grid for the pristine cell. A supercell of size n uses SCF_KGRID / n, so the
# k-point density is the same at every size. If not set, KPPRA is used by default.
SCF_KGRID = None  # e.g. [8, 8, 8]

# Whose total_energy and band_gaps properties to reuse for the pristine supercells:
# "public" (any owner, highest precision wins), "curators" (only curators'),
# or "my_account" (curators' or your own).
PRISTINE_PROPERTY_SOURCE = "my_account"

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Build the supercell pairs
### 3.1. Load the pristine cell

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

pristine = load_material_from_folder(FOLDER, PRISTINE_NAME) or Material.create(
    Materials.get_by_name_first_match(PRISTINE_NAME)
)

visualize_materials(pristine, repetitions=VISUALIZATION_REPETITIONS, title="Pristine cell")

### 3.2. Build a pristine/defective pair per supercell size
The pristine cell is scaled first and the defects are placed into each supercell, so that every size holds the same isolated defect at the same site. Defect coordinates are given in the pristine cell's crystal basis and are divided by the scaling to reach that site in the supercell.

Atom labels are stripped: they are useful for analysis notebooks but are not compatible with QE input generation.

In [ ]:
from mat3ra.made.tools.helpers import PointDefectDict, create_multiple_defects, create_supercell


def build_pair(scaling):
    supercell = create_supercell(pristine, scaling_factor=[scaling] * 3)
    defect_dicts = [
        PointDefectDict(**{**config, "coordinate": [x / scaling for x in config["coordinate"]]})
        for config in DEFECT_CONFIGS
    ]
    defective = create_multiple_defects(material=supercell, defect_dicts=defect_dicts)
    for material, kind in ((supercell, "pristine"), (defective, "defective")):
        material.basis.set_labels_from_list([])
        material.name = f"{pristine.name} {scaling}x{scaling}x{scaling} {kind}"
    return supercell, defective


pairs = {scaling: build_pair(scaling) for scaling in SUPERCELL_SCALINGS}

visualize_materials(
    [{"material": material, "title": material.name} for pair in pairs.values() for material in pair],
    repetitions=VISUALIZATION_REPETITIONS,
    rotation="-90x",
)

### 3.3. Resolve Standata elemental reference materials
Elemental chemical potentials come from Standata materials tagged `elemental` with `metadata.element`. Each must also have a refined `total_energy` -- this is enforced by the workflow at runtime; run [Total Energy](total_energy.ipynb) for any elemental reference that is missing one.

The chemical-potential term covers every species whose count changes, so elements are taken from the union of the pristine and defective structures.

In [ ]:
elements = sorted(
    {element for pair in pairs.values() for material in pair for element in material.basis.elements.values}
)
elemental_materials_data = client.materials.list(
    {"tags": "elemental", "metadata.element": {"$in": elements}},
)
available = {data.get("metadata", {}).get("element") for data in elemental_materials_data}
missing = sorted(set(elements) - available)
if missing:
    raise RuntimeError(
        f"Missing elemental reference material(s) for {missing}. "
        "Add elemental material(s) from Standata, or add tag 'elemental' with metadata.element = {element}"
    )
print(f"Resolved elemental reference materials for: {', '.join(elements)}")

### 3.4. Save the supercells to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pairs = {
    scaling: tuple(Material.create(get_or_create_material(client, material, ACCOUNT_ID)) for material in pair)
    for scaling, pair in pairs.items()
}
for scaling, (saved_pristine, saved_defective) in saved_pairs.items():
    print(f"✅ n={scaling}: pristine {saved_pristine.id} ({len(saved_pristine.basis.elements.ids)} atoms), "
          f"defective {saved_defective.id} ({len(saved_defective.basis.elements.ids)} atoms)")

## 4. Configure the workflows
### 4.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load the Defect Formation Energy workflow and apply size and charge
One workflow per (size, charge): the charge is written into the `&SYSTEM` namelist of the defective-cell SCF, and the k-grid is scaled down with the supercell to hold the k-point density fixed. The first combination is previewed below.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid, patch_workflow_qe_input

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)


def scf_kgrid_for(scaling):
    return None if SCF_KGRID is None else [max(1, dimension // scaling) for dimension in SCF_KGRID]


def defect_workflow_for(scaling, charge):
    workflow = Workflow.create(defect_workflow_config)
    workflow.name = f"{MY_WORKFLOW_NAME} n={scaling} q={charge:+d}"
    if charge:
        patch_workflow_qe_input(workflow, {"system": {"tot_charge": charge}}, unit_names=["pw_scf"])
    return apply_scf_kgrid(workflow, scf_kgrid_for(scaling), material=saved_pairs[scaling][1])


visualize_workflow(defect_workflow_for(SUPERCELL_SCALINGS[0], CHARGES[0]))

### 4.3. Load the pristine reference workflows
Each pristine supercell supplies two references: its total energy, which the workflow subtracts, and its band gap, which supplies $E_{\text{VBM}}$ and the range of $\mu_e$. Both are reused when the material already carries them and computed otherwise.

In [ ]:
pristine_workflow_configs = {
    property_name: WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(search_term)
    for property_name, search_term in {"total_energy": "total_energy.json", "band_gaps": "band_gap.json"}.items()
}
print(f"Loaded pristine reference workflows: {', '.join(pristine_workflow_configs)}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create and run the jobs
### 6.1. Create and submit the pristine reference jobs

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.core.entity.property.api import find_property_for_material
from mat3ra.notebooks_utils.job import create_job


def create_job_for(materials, workflow):
    job = create_job(
        api_client=client,
        materials=materials,
        workflow=workflow,
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{workflow.name} {timestamp}",
        compute=compute.to_dict(),
    )
    return job[0] if isinstance(job, list) else job


pristine_job_ids = []
for scaling, (saved_pristine, _) in saved_pairs.items():
    for property_name, workflow_config in pristine_workflow_configs.items():
        if find_property_for_material(client, saved_pristine.id, property_name, PRISTINE_PROPERTY_SOURCE):
            print(f"♻️  n={scaling}: reusing the existing {property_name} of {saved_pristine.name}")
            continue
        workflow = Workflow.create(workflow_config)
        workflow.name = f"{workflow.name} {saved_pristine.name}"
        apply_scf_kgrid(workflow, scf_kgrid_for(scaling), material=saved_pristine)
        job = create_job_for([saved_pristine], workflow)
        pristine_job_ids.append(job["_id"])
        print(f"✅ n={scaling}: created a {property_name} job {job['_id']}")

if pristine_job_ids:
    submit_jobs(client.jobs, pristine_job_ids)
    print(f"✅ Submitted {len(pristine_job_ids)} pristine reference job(s).")

In [ ]:
if pristine_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, pristine_job_ids, poll_interval=POLL_INTERVAL)

### 6.2. Create the Defect Formation Energy jobs, one per size and charge

In [ ]:
import pandas as pd

job_records = []
for scaling in SUPERCELL_SCALINGS:
    saved_pristine, saved_defective = saved_pairs[scaling]
    for charge in CHARGES:
        # Order matters: [0] defective (computed), [1] pristine (reference).
        job = create_job_for([saved_defective, saved_pristine], defect_workflow_for(scaling, charge))
        job_records.append({
            "scaling": scaling,
            "charge": charge,
            "atoms": len(saved_defective.basis.elements.ids),
            "length": saved_pristine.lattice.cell_volume ** (1 / 3),
            "job_id": job["_id"],
        })

job_ids = [record["job_id"] for record in job_records]
pd.DataFrame(job_records)

### 6.3. Submit the jobs and monitor the statuses

In [ ]:
submit_jobs(client.jobs, job_ids)
print(f"✅ Submitted {len(job_ids)} Defect Formation Energy jobs successfully!")

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 7. Retrieve results
### 7.1. Raw formation energies
`length` is $L = V^{1/3}$ of the supercell, the length scale the image-charge terms in section 7.3 are expressed in.

In [ ]:
results_records = []
for record in job_records:
    properties = client.properties.get_for_job(record["job_id"], property_name="defect_formation_energy")
    results_records.append({**record, "raw_formation_energy": properties[0]["value"] if properties else None})

results_df = pd.DataFrame(results_records)
results_df

### 7.2. Formation energy versus the electron chemical potential
Each charge state gives a straight line of slope $q$ over $\mu_e \in [0, E_{\text{gap}}]$; the lower envelope is the charge state the defect actually adopts, and the crossings between the lines are the charge transition levels. The largest supercell is used, with $E_{\text{VBM}}$ and $E_{\text{gap}}$ from its own Band Gap job.

In [ ]:
import numpy as np
import plotly.graph_objects as go

largest_scaling = max(SUPERCELL_SCALINGS)
band_gaps = find_property_for_material(
    client, saved_pairs[largest_scaling][0].id, "band_gaps", PRISTINE_PROPERTY_SOURCE
)["data"]
fundamental_gap = min(band_gaps["values"], key=lambda entry: entry["value"])
valence_band_maximum, band_gap = fundamental_gap["eigenvalueValence"], fundamental_gap["value"]
print(f"Pristine n={largest_scaling}: VBM {valence_band_maximum:.4f} eV, "
      f"{fundamental_gap['type']} gap {band_gap:.4f} eV")

electron_chemical_potential = np.linspace(0, band_gap, 201)
largest_df = results_df[results_df["scaling"] == largest_scaling]
formation_energies = {
    row.charge: row.raw_formation_energy + row.charge * (valence_band_maximum + electron_chemical_potential)
    for row in largest_df.itertuples()
}

figure = go.Figure()
for charge, energies in sorted(formation_energies.items()):
    figure.add_scatter(x=electron_chemical_potential, y=energies, mode="lines", name=f"q = {charge:+d}")
figure.add_scatter(
    x=electron_chemical_potential,
    y=np.min(list(formation_energies.values()), axis=0),
    mode="lines",
    name="stable state",
    line={"width": 8, "color": "black"},
    opacity=0.2,
)
figure.update_layout(
    title=f"Defect formation energy vs electron chemical potential (n={largest_scaling})",
    xaxis_title="Electron chemical potential above VBM (eV)",
    yaxis_title="Formation energy (eV)",
)
figure.show()

stable_charges = np.array(
    [
        min(formation_energies, key=lambda charge: formation_energies[charge][index])
        for index in range(len(electron_chemical_potential))
    ]
)
for charge, energies in sorted(formation_energies.items()):
    window = electron_chemical_potential[stable_charges == charge]
    stability = f"stable for mu_e in [{window.min():.3f}, {window.max():.3f}] eV" if window.size else "never stable"
    print(f"q = {charge:+d}: E_f = {energies[0]:.4f} eV at mu_e = 0 (VBM), {stability}")

### 7.3. Finite-size extrapolation
The raw formation energy of a charged cell carries the spurious interaction of the defect with its periodic images, which falls off as $a/L + b/L^3$. Fitting the sizes in `SUPERCELL_SCALINGS` extrapolates to the isolated defect $E_\infty$, in place of an analytical correction; $b$ needs three or more sizes.

In [ ]:
def extrapolate(lengths, energies):
    terms = [np.ones_like(lengths), 1 / lengths] + ([1 / lengths**3] if len(lengths) > 2 else [])
    coefficients, *_ = np.linalg.lstsq(np.stack(terms, axis=1), energies, rcond=None)
    return dict(zip(("E_inf", "a", "b"), coefficients))


fit_records = []
figure = go.Figure()
for charge, group in results_df.groupby("charge"):
    lengths, energies = group["length"].to_numpy(), group["raw_formation_energy"].to_numpy()
    if len(lengths) < 2:
        print(f"q = {charge:+d}: E_f = {energies[0]:.4f} eV at n={group['scaling'].iloc[0]}; "
              "add sizes to SUPERCELL_SCALINGS to extrapolate to the isolated defect.")
        continue
    fit = extrapolate(lengths, energies)
    fit_records.append({"charge": charge, **fit})
    inverse_lengths = np.linspace(0, 1 / lengths.min(), 50)
    figure.add_scatter(x=1 / lengths, y=energies, mode="markers", name=f"q = {charge:+d}")
    figure.add_scatter(
        x=inverse_lengths,
        y=fit["E_inf"] + fit["a"] * inverse_lengths + fit.get("b", 0.0) * inverse_lengths**3,
        mode="lines",
        showlegend=False,
    )

if fit_records:
    figure.update_layout(
        title="Finite-size extrapolation of the defect formation energy",
        xaxis_title="1 / L (1/Angstrom), L = V^(1/3)",
        yaxis_title="Raw formation energy (eV)",
    )
    figure.show()

pd.DataFrame(fit_records)